# ECOM060 — Instrumentação Eletrônica (2026.2)## Caracterização do Sistema Massa–Mola–Amortecedor**Universidade Federal de Alagoas** · Prof. Maurício Beltrão Rossiter**Equipe:** Pedro Henrique Vieira Giló · Thiago Fellype Marques Laurentino · Caio Oliveira França dos Anjos---Este notebook gera **todos** os resultados numéricos e **todas as figuras** do relatório decaracterização. Ele roda no Google Colab sem instalar nada (usa apenas `numpy`, `scipy` e `matplotlib`).**Modelo:**$$m\,\ddot{x}(t) + c\,\dot{x}(t) + k\,x(t) = F(t)$$**Como usar:**1. `Ambiente de execução → Executar tudo` (ou Ctrl+F9)2. A última célula compacta as 7 figuras em `figuras_relatorio.zip` e faz o download3. Faça upload dos `.png` na raiz do projeto do Overleaf, junto do `main.tex`

## 1. Importações e configuração

In [ ]:
import numpy as npfrom scipy.integrate import solve_ivpfrom scipy.signal import argrelextremaimport matplotlib.pyplot as pltplt.rcParams.update({    "figure.dpi": 110,    "savefig.dpi": 200,    "font.size": 10,    "axes.grid": True,    "grid.alpha": 0.3,})# paleta usada nas figuras do relatorioAZUL, AMARELO, VERDE, VERMELHO, CINZA = "#1f6f8b", "#e0a800", "#2ca02c", "#d62728", "#888888"print("numpy", np.__version__)

## 2. Parâmetros do sistemaValores da Seção 5 do relatório. `c` foi o parâmetro ajustado para fixar $\zeta = 0{,}30$(regime subamortecido); `m` e `k` foram escolhidos primeiro, produzindo $\omega_n = 10$ rad/s exatos.

In [ ]:
m    = 1.0     # kg          - massac    = 6.0     # N.s/m       - coeficiente de amortecimento viscosok    = 100.0   # N/m         - rigidez da molaF0   = 10.0    # N           - forca nominal de ensaio (degrau)xmax = 0.20    # m           - curso maximo admissivel (dominio de validade)x_inf = F0 / k          # deslocamento de regime permanenteprint(f"x_inf = F0/k = {x_inf:.4f} m")

## 3. Espaço de estados e caracterização dinâmicaCom $x_1 = x$, $x_2 = \dot{x}$ e $u = F(t)$:$$A = \begin{bmatrix} 0 & 1 \\ -k/m & -c/m \end{bmatrix}, \qquadB = \begin{bmatrix} 0 \\ 1/m \end{bmatrix}, \qquadC = \begin{bmatrix} 1 & 0 \end{bmatrix}$$

In [ ]:
A = np.array([[0.0,   1.0],              [-k/m, -c/m]])B = np.array([[0.0], [1.0/m]])C = np.array([[1.0, 0.0]])# --- via analitica (formulas do Guia de Caracterizacao) ---wn   = np.sqrt(k/m)                 # frequencia natural [rad/s]zeta = c/(2*np.sqrt(k*m))           # coeficiente de amortecimento [-]wd   = wn*np.sqrt(1 - zeta**2)      # frequencia natural amortecida [rad/s]tau  = 1/(zeta*wn)                  # constante de tempo dominante [s]Kdc  = 1/k                          # ganho DC [m/N]Td   = 2*np.pi/wd                   # periodo de oscilacao [s]OS   = np.exp(-zeta*np.pi/np.sqrt(1 - zeta**2))   # overshoot [-]Tp   = np.pi/wd                     # tempo de pico [s]Tr   = 1.8/wn                       # tempo de subida aprox. [s]Ts2  = 4/(zeta*wn)                  # acomodacao 2% [s]Ts5  = 3/(zeta*wn)                  # acomodacao 5% [s]# --- VERIFICACAO CRUZADA OBRIGATORIA (Requisito 8): via numerica ---lam = np.linalg.eigvals(A)wn_num   = np.sqrt(np.linalg.det(A))zeta_num = -lam[0].real/wn_numstiff    = max(abs(lam))/min(abs(lam))print("=== CARACTERIZACAO DINAMICA ===")print(f"Polos (eigvals)      : {lam[0]:.6f} / {lam[1]:.6f} rad/s")print(f"wn   analitico/numerico : {wn:.6f} / {wn_num:.6f} rad/s")print(f"zeta analitico/numerico : {zeta:.6f} / {zeta_num:.6f}")print(f"wd                   : {wd:.6f} rad/s   (fd = {wd/(2*np.pi):.4f} Hz)")print(f"Td (periodo)         : {Td:.6f} s")print(f"K (ganho DC)         : {Kdc:.6f} m/N")print(f"tau                  : {tau:.6f} s")print(f"Overshoot            : {OS*100:.4f} %")print(f"Tp                   : {Tp:.6f} s")print(f"Tr (aprox 1.8/wn)    : {Tr:.6f} s")print(f"Ts 2% (4*tau)        : {Ts2:.6f} s")print(f"Ts 5% (3*tau)        : {Ts5:.6f} s")print(f"Stiffness ratio      : {stiff:.4f}")print()if zeta < 0:            print("Classificacao: INSTAVEL")elif np.isclose(zeta,0):print("Classificacao: NAO AMORTECIDO")elif zeta < 1:          print("Classificacao: SUBAMORTECIDO (polos complexos conjugados)")elif np.isclose(zeta,1):print("Classificacao: CRITICAMENTE AMORTECIDO")else:                   print("Classificacao: SOBREAMORTECIDO")

## 4. Soluções algébricas de referênciaPor ser LTI, o sistema admite solução fechada nos três cenários — o que permite validaçãocruzada em todos eles (e não só em um, como aconteceria num sistema não linear).

In [ ]:
# Resposta ao degrau F*u(t) com condicoes iniciais nulas (Eq. 15 do relatorio)def x_step(t, F=F0):    t = np.asarray(t, dtype=float)    e = np.exp(-zeta*wn*t)    return (F/k)*(1 - e*(np.cos(wd*t) + (zeta/np.sqrt(1-zeta**2))*np.sin(wd*t)))# Resposta livre com x(0)=x0, xdot(0)=v0 (Eq. 13 do relatorio)def x_free(t, x0, v0=0.0):    t = np.asarray(t, dtype=float)    e = np.exp(-zeta*wn*t)    return e*(x0*np.cos(wd*t) + ((v0 + zeta*wn*x0)/wd)*np.sin(wd*t))# Envelope de decaimento da resposta livre para v0 = 0def envelope(t, x0):    return x0*np.sqrt(1 + (zeta*wn/wd)**2)*np.exp(-zeta*wn*np.asarray(t, dtype=float))# EDO em espaco de estados: y = [x, xdot]def modelo(t, y, F_func):    x, v = y    return [v, (F_func(t) - c*v - k*x)/m]RTOL, ATOL = 1e-11, 1e-13   # tolerancias do RK45 usadas em todos os cenarios

## 5. Cenário 1 — Ensaio ao degrau a partir do repouso$x(0) = 0$, $\dot{x}(0) = 0$, degrau de força para $F_0 = 10$ N em $t = 0$.

In [ ]:
T1 = 2.5t1 = np.linspace(0, T1, 20001)sol1 = solve_ivp(modelo, [0, T1], [0.0, 0.0], t_eval=t1,                 method="RK45", rtol=RTOL, atol=ATOL, args=(lambda t: F0,))x1_num = sol1.y[0]x1_alg = x_step(t1)err1   = np.abs(x1_num - x1_alg)# pontos caracteristicos MEDIDOS na curvaipk    = np.argmax(x1_num)OS_med = (x1_num[ipk] - x_inf)/x_inffora   = np.where(np.abs(x1_num - x_inf) > 0.02*x_inf)[0]Ts_med = t1[fora[-1] + 1]i10 = np.argmax(x1_num >= 0.10*x_inf)i90 = np.argmax(x1_num >= 0.90*x_inf)Tr_med = t1[i90] - t1[i10]print(f"x_inf            : {x1_num[-1]:.6f} m  (teorico {x_inf:.6f} m)")print(f"Pico             : {x1_num[ipk]:.6f} m em t = {t1[ipk]:.4f} s")print(f"Overshoot medido : {OS_med*100:.4f} %   (teorico {OS*100:.4f} %)")print(f"Tp medido        : {t1[ipk]:.5f} s      (teorico {Tp:.5f} s)")print(f"Ts(2%) medido    : {Ts_med:.4f} s       (aprox. 4*tau = {Ts2:.4f} s)")print(f"Tr(10-90) medido : {Tr_med:.4f} s       (aprox. 1.8/wn = {Tr:.4f} s)")print(f"ERRO MAX num x alg: {err1.max():.3e} m")

In [ ]:
# ---- Figura: fig_cen1_degrau.png ----fig, ax = plt.subplots(figsize=(7.2, 4.4))ax.plot(t1, x1_num, lw=2, color=AZUL, label="x(t) — simulação numérica (RK45)")ax.axhline(x_inf, ls="--", color=AMARELO, label=f"$x_\\infty$ = {x_inf:.3f} m (regime permanente)")ax.axhspan(0.98*x_inf, 1.02*x_inf, color=VERDE, alpha=0.12, label="faixa de $\\pm$2%")ax.plot(t1[ipk], x1_num[ipk], "o", color=VERMELHO, ms=7)ax.annotate(f"$T_p$={t1[ipk]:.3f} s\novershoot={OS_med*100:.1f}%",            xy=(t1[ipk], x1_num[ipk]), xytext=(t1[ipk]+0.45, x1_num[ipk]-0.004),            arrowprops=dict(arrowstyle="->", color=VERMELHO), color=VERMELHO, fontsize=9)ax.axvline(Ts_med, ls=":", color="k", lw=1)ax.text(Ts_med+0.03, 0.045, f"$T_s$(2%)={Ts_med:.3f} s", fontsize=9)ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Deslocamento x (m)")ax.set_title("Cenário 1 — Ensaio ao degrau a partir do repouso")ax.set_ylim(-0.005, 0.155); ax.legend(loc="lower right", fontsize=8)fig.tight_layout(); fig.savefig("fig_cen1_degrau.png"); plt.show()

In [ ]:
# ---- Figura: fig_cen1_erro.png ----fig, ax = plt.subplots(figsize=(7.2, 3.0))ax.plot(t1, err1, color=AZUL, lw=1)ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Erro absoluto |x_num - x_alg| (m)")ax.set_title("Cenário 1 — Erro entre solução numérica e algébrica")fig.tight_layout(); fig.savefig("fig_cen1_erro.png"); plt.show()

## 6. Cenário 2 — Condição inicial não nula, sem entrada (resposta livre)$x(0) = 0{,}10$ m, $\dot{x}(0) = 0$, $F = 0$ durante todo o ensaio.É deste ensaio que se extrai $\zeta$ experimentalmente, por **decremento logarítmico**.

In [ ]:
x0, v0 = 0.10, 0.0T2 = 2.5t2 = np.linspace(0, T2, 20001)sol2 = solve_ivp(modelo, [0, T2], [x0, v0], t_eval=t2,                 method="RK45", rtol=RTOL, atol=ATOL, args=(lambda t: 0.0,))x2_num, v2_num = sol2.y[0], sol2.y[1]x2_alg = x_free(t2, x0, v0)err2   = np.abs(x2_num - x2_alg)print(f"ERRO MAX num x alg: {err2.max():.3e} m")# ---- Identificacao de zeta por decremento logaritmico ----picos = argrelextrema(x2_num, np.greater)[0]p1, p2 = picos[0], picos[1]delta     = np.log(x2_num[p1]/x2_num[p2])zeta_est  = delta/np.sqrt(4*np.pi**2 + delta**2)Td_med    = t2[p2] - t2[p1]wd_est    = 2*np.pi/Td_medc_est     = 2*zeta_est*np.sqrt(k*m)print("\n--- Decremento logaritmico ---")print(f"pico 1: x = {x2_num[p1]:.7f} m em t = {t2[p1]:.5f} s")print(f"pico 2: x = {x2_num[p2]:.7f} m em t = {t2[p2]:.5f} s")print(f"delta      = {delta:.5f}")print(f"zeta_est   = {zeta_est:.6f}   (verdadeiro {zeta:.6f})")print(f"Td medido  = {Td_med:.5f} s -> wd_est = {wd_est:.4f} rad/s (teorico {wd:.5f})")print(f"c estimado = {c_est:.5f} N.s/m  (verdadeiro {c:.5f})")

In [ ]:
# ---- Figura: fig_cen2_livre.png ----fig, ax = plt.subplots(figsize=(7.2, 4.2))ax.plot(t2, x2_alg, lw=3, color=AMARELO, label="Solução algébrica")ax.plot(t2, x2_num, "--", lw=1.6, color=AZUL, label="Solução numérica (RK45)")ax.plot(t2,  envelope(t2, x0), ":", color=CINZA, lw=1.2,        label="Envelope $\\pm x_0\\sqrt{1+(\\zeta\\omega_n/\\omega_d)^2}e^{-\\zeta\\omega_n t}$")ax.plot(t2, -envelope(t2, x0), ":", color=CINZA, lw=1.2)ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Deslocamento x (m)")ax.set_title("Cenário 2 — Resposta livre: numérico $\\times$ algébrico")ax.legend(fontsize=8)fig.tight_layout(); fig.savefig("fig_cen2_livre.png"); plt.show()

In [ ]:
# ---- Figura: fig_cen2_erro.png ----fig, ax = plt.subplots(figsize=(7.2, 3.0))ax.plot(t2, err2, color=AZUL, lw=1)ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Erro absoluto |x_num - x_alg| (m)")ax.set_title("Cenário 2 — Erro entre solução numérica e algébrica")fig.tight_layout(); fig.savefig("fig_cen2_erro.png"); plt.show()

In [ ]:
# ---- Figura: fig_cen2_fase.png (retrato de fase) ----fig, ax = plt.subplots(figsize=(5.4, 4.6))ax.plot(x2_num, v2_num, color=AZUL, lw=1.5)ax.plot(x0, v0, "o", color=VERDE, ms=7, label="início (x$_0$=0,10 m)")ax.plot(0, 0, "X", color=VERMELHO, ms=10, label="equilíbrio (0,0)")ax.set_xlabel("Posição x (m)"); ax.set_ylabel("Velocidade $\\dot{x}$ (m/s)")ax.set_title("Cenário 2 — Retrato de fase")ax.legend(fontsize=8)fig.tight_layout(); fig.savefig("fig_cen2_fase.png"); plt.show()

## 7. Cenário 3 — Transitório temporário (pulso) e recuperaçãoParte do regime permanente ($x_0 = 0{,}100$ m sob $F_0$) e recebe $F = 1{,}5\,F_0$entre $t = 2{,}0$ s e $t = 2{,}5$ s.A validação aqui usa **superposição** de duas respostas ao degrau defasadas — o que, além devalidar o cenário, verifica numericamente a própria linearidade do modelo implementado.

In [ ]:
t_on, t_off = 2.0, 2.5dF = 0.5*F0          # acrescimo durante o pulsoT3 = 6.0t3 = np.linspace(0, T3, 30001)def F_pulso(t):    return F0 + (dF if (t_on <= t < t_off) else 0.0)sol3 = solve_ivp(modelo, [0, T3], [x_inf, 0.0], t_eval=t3, method="RK45",                 rtol=RTOL, atol=ATOL, max_step=1e-3, args=(F_pulso,))x3_num = sol3.y[0]def degrau_deslocado(t, t0, F):    y = np.zeros_like(t); msk = t >= t0    y[msk] = x_step(t[msk] - t0, F)    return yx3_alg = x_inf + degrau_deslocado(t3, t_on, dF) - degrau_deslocado(t3, t_off, dF)err3   = np.abs(x3_num - x3_alg)ipk3 = np.argmax(x3_num)apos = t3 > t_offivl3 = np.argmax(apos) + np.argmin(x3_num[apos])frac = np.where(np.abs(x3_num - x_inf) > 0.02*x_inf)[0]t_rec = t3[frac[-1] + 1]print(f"Pico       : {x3_num[ipk3]:.6f} m em t = {t3[ipk3]:.4f} s")print(f"Vale       : {x3_num[ivl3]:.6f} m em t = {t3[ivl3]:.4f} s")print(f"Retorno a faixa de 2%: t = {t_rec:.4f} s  ({t_rec - t_off:.4f} s apos o fim do pulso)")print(f"x(t=6 s)   : {x3_num[-1]:.6f} m  (desvio residual {abs(x3_num[-1]-x_inf):.2e} m)")print(f"ERRO MAX vs superposicao: {err3.max():.3e} m")print(f"Deslocamento maximo {x3_num[ipk3]:.4f} m vs limite {xmax:.2f} m -> "      f"margem de {100*(1 - x3_num[ipk3]/xmax):.1f} %")

In [ ]:
# ---- Figura: fig_cen3_pulso.png ----fig, ax = plt.subplots(figsize=(7.2, 4.2))ax.plot(t3, x3_num, lw=1.8, color=AZUL, label="x(t) — resposta ao pulso de força")ax.axhline(x_inf, ls="--", color=AMARELO, label=f"$x_0$ = {x_inf:.3f} m")ax.axvspan(t_on, t_off, color="#7ec8e3", alpha=0.35, label="pulso ativo (F = 1,5 F$_0$)")ax.plot(t3[ipk3], x3_num[ipk3], "o", color=VERMELHO, ms=6)ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Deslocamento x (m)")ax.set_title("Cenário 3 — Transitório temporário (pulso) e recuperação")ax.legend(fontsize=8)fig.tight_layout(); fig.savefig("fig_cen3_pulso.png"); plt.show()

## 8. Mapa de polos

In [ ]:
# ---- Figura: fig_mapa_polos.png ----fig, ax = plt.subplots(figsize=(5.4, 4.8))th = np.linspace(0, 2*np.pi, 400)ax.plot(wn*np.cos(th), wn*np.sin(th), "--", color=VERDE, lw=1,        label=f"$|\\lambda| = \\omega_n$ = {wn:.1f} rad/s")ax.plot([0, -wn*zeta*1.35], [0,  wd*1.35], ":", color="k", lw=1,        label=f"$\\zeta$ = {zeta:.2f} (ângulo constante)")ax.plot([0, -wn*zeta*1.35], [0, -wd*1.35], ":", color="k", lw=1)ax.plot(lam.real, lam.imag, "X", color=VERMELHO, ms=13, label="polos")ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8)ax.set_xlabel("Re($\\lambda$) [rad/s]"); ax.set_ylabel("Im($\\lambda$) [rad/s]")ax.set_title("Mapa de polos — massa-mola-amortecedor")ax.set_aspect("equal"); ax.legend(fontsize=8, loc="lower left")fig.tight_layout(); fig.savefig("fig_mapa_polos.png"); plt.show()print(f"Angulo com o eixo real negativo: arccos(zeta) = {np.degrees(np.arccos(zeta)):.2f} graus")

## 9. Requisito 9 — representação numérica e escolha do passoRepetição, para este sistema, do experimento que o docente conduziu no RLC série: integrar oCenário 1 por **Euler explícito**, variando o passo $h$ e a precisão da representação, e medir oerro máximo contra a solução analítica.O ponto de interesse: em `float64` o refino de $h$ sempre ajuda; em `float32` ele **deixa de ajudare passa a atrapalhar**, porque cada passo carrega um arredondamento e mais passos significam maiseventos de arredondamento acumulados.> Esta célula é a mais lenta do notebook (~1 min), por causa do laço de Euler em Python puro com> $h = 10^{-5}$ s.

In [ ]:
# Euler explicito do Cenario 1, na precisao pedida; retorna o erro maximo vs solucao exatadef euler_erro_max(h, dtype, T=2.5):    n  = int(round(T/h))    y  = np.array([0.0, 0.0], dtype=dtype)    hh = dtype(h)    mm, cc, kk, FF = dtype(m), dtype(c), dtype(k), dtype(F0)    emax, tt = 0.0, dtype(0.0)    for _ in range(n):        a = (FF - cc*y[1] - kk*y[0])/mm        y = np.array([y[0] + hh*y[1], y[1] + hh*a], dtype=dtype)        tt = dtype(tt + hh)        e = abs(float(y[0]) - float(x_step(float(tt))))        if e > emax:            emax = e    return emaxprint(f"{'h [s]':>10} | {'float64':>14} | {'float32':>14}")print("-"*46)tabela = []for h in [1e-2, 1e-3, 1e-4, 1e-5]:    e64 = euler_erro_max(h, np.float64)    e32 = euler_erro_max(h, np.float32)    tabela.append((h, e64, e32))    print(f"{h:>10.0e} | {e64:>14.3e} | {e32:>14.3e}")print("\nCritério de passo pela dinâmica: wn*h <= 0.01  ->  h <=", f"{0.01/wn*1000:.2f} ms")

## 10. Resumo — Ficha do SistemaImpressão consolidada de todos os valores citados no relatório.

In [ ]:
print("="*66)print(" FICHA DO SISTEMA - MASSA-MOLA-AMORTECEDOR (ECOM060 2026.2)")print("="*66)print(f" Ordem                      : 2a")print(f" Classificacao              : SISO, linear, invariante, subamortecido, estavel")print(f" Parametros                 : m={m} kg, c={c} N.s/m, k={k} N/m")print(f" Polos                      : {lam[0].real:+.6f} +/- j{abs(lam[0].imag):.6f} rad/s")print(f" wn / zeta                  : {wn:.6f} rad/s / {zeta:.6f}")print(f" wd / Td                    : {wd:.6f} rad/s / {Td:.6f} s")print(f" Ganho DC K                 : {Kdc:.6f} m/N")print(f" tau dominante              : {tau:.6f} s ({tau*1000:.1f} ms)")print(f" Overshoot / Tp             : {OS*100:.4f} % / {Tp:.6f} s")print(f" Ts(2%) formula / medido    : {Ts2:.4f} s / {Ts_med:.4f} s")print(f" Tr formula / medido        : {Tr:.4f} s / {Tr_med:.4f} s")print(f" Stiffness ratio            : {stiff:.4f} (nao stiff)")print(f" Dominio de validade        : |x| <= {xmax:.2f} m")print("-"*66)print(f" Erro max Cen.1 (num x alg) : {err1.max():.3e} m")print(f" Erro max Cen.2 (num x alg) : {err2.max():.3e} m")print(f" Erro max Cen.3 (superpos.) : {err3.max():.3e} m")print(f" zeta por decremento log.   : {zeta_est:.6f} (verdadeiro {zeta:.6f})")print("="*66)

## 11. Download das figurasCompacta os 7 PNGs e faz o download. Descompacte e faça upload dos arquivos na **raiz** do projetodo Overleaf, junto do `main.tex`.

In [ ]:
import zipfile, osfiguras = ["fig_cen1_degrau.png", "fig_cen1_erro.png",           "fig_cen2_livre.png",  "fig_cen2_erro.png", "fig_cen2_fase.png",           "fig_cen3_pulso.png",  "fig_mapa_polos.png"]faltando = [f for f in figuras if not os.path.exists(f)]if faltando:    print("ATENCAO - figuras nao geradas:", faltando)else:    with zipfile.ZipFile("figuras_relatorio.zip", "w") as z:        for f in figuras:            z.write(f)    print("figuras_relatorio.zip criado com", len(figuras), "figuras.")    try:        from google.colab import files        files.download("figuras_relatorio.zip")    except ImportError:        print("(fora do Colab: o arquivo esta no diretorio de trabalho)")